# Phase 4: Deep Learning Models — Smoke Test

Runs a forward-pass + 1 training step on tiny data to verify everything compiles.
Full training on remote server.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
from pathlib import Path

from src import (
    load_articles, load_categories, load_states_train, load_states_test,
    load_or_build_adjacency
)
from src.dl_model import (
    MLPScorer, GCNScorer, CandidateDataset, build_normalized_adj, predict
)

BASE = Path('/home/agung/Code/datathon/datathon-kaggle-task')
CACHE = BASE / '.cache'
DATA = BASE / 'dataset-task2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

articles = load_articles()
categories = load_categories()
train = load_states_train()
test = load_states_test()
adj = load_or_build_adjacency()
print(f'Articles: {len(articles)}, Train: {len(train)}, Test: {len(test)}')

In [ ]:
# Load precomputed embeddings
embeddings = np.load(CACHE / 'article_embeddings.npy')

all_cats = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(all_cats)}
cat_enc = np.zeros((len(articles), len(all_cats)), dtype=np.float32)
for _, r in categories.iterrows():
    cat_enc[r['article_id'], cat_to_idx[r['category']]] = 1.0

n_cats = cat_enc.shape[1]
print(f'Embeddings: {embeddings.shape}, Categories: {cat_enc.shape}')

## Smoke Test 1: MLPScorer

Forward pass + 1 train step on 50 states.

In [ ]:
tiny = train.sample(50, random_state=42)
dataset = CandidateDataset(tiny, adj, embeddings, cat_enc)
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

model_mlp = MLPScorer(emb_dim=384, n_cats=n_cats, hidden_dim=128).to(device)
optimizer = torch.optim.Adam(model_mlp.parameters(), lr=1e-3)
loss_fn = torch.nn.BCEWithLogitsLoss()

# Forward pass + 1 step
model_mlp.train()
for batch in loader:
    curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats, out_deg, n_links, labels = [
        x.to(device) for x in batch
    ]
    scores = model_mlp(curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats, out_deg, n_links)
    loss = loss_fn(scores, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'MLPScorer: batch loss = {loss.item():.4f}, scores shape = {scores.shape}')
    break

total_params = sum(p.numel() for p in model_mlp.parameters())
print(f'MLPScorer params: {total_params:,}')

## Smoke Test 2: GCNScorer

Build normalized adjacency and run GCN forward pass.

In [ ]:
n_nodes = len(articles)
adj_norm = build_normalized_adj(adj, n_nodes, device=device)
print(f'Adj norm shape: {adj_norm.shape}')

model_gnn = GCNScorer(node_dim=384, hidden_dim=128, out_dim=64, n_cats=n_cats).to(device)
optimizer = torch.optim.Adam(model_gnn.parameters(), lr=1e-3)

# GCN batch: process all node features at once, then score (state, candidate)
model_gnn.train()
all_emb = torch.tensor(embeddings, dtype=torch.float, device=device)
all_cats = torch.tensor(cat_enc, dtype=torch.float, device=device)

for batch in loader:
    curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats, out_deg, n_links, labels = [
        x.to(device) for x in batch
    ]
    # Resolve node IDs for this batch
    batch_node_ids = []
    # We stored everything as values, but we need node IDs for GCN lookup
    # For smoke test, just use a fixed set
    node_ids = torch.zeros((curr_emb.size(0), 3), dtype=torch.long, device=device)
    scores = model_gnn(curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats,
                       out_deg, n_links, adj_norm, node_ids)
    loss = loss_fn(scores, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    print(f'GCNScorer: batch loss = {loss.item():.4f}, scores shape = {scores.shape}')
    break

total_params = sum(p.numel() for p in model_gnn.parameters())
print(f'GCNScorer params: {total_params:,}')

## Smoke Test 3: Prediction

Test the `predict()` function on 10 test states.

In [ ]:
test_tiny = test.head(10)

preds_mlp = predict(model_mlp, test_tiny, adj, embeddings, cat_enc,
                    adj_norm=None, device=device)
print(f'MLP predictions: {preds_mlp}')

preds_gnn = predict(model_gnn, test_tiny, adj, embeddings, cat_enc,
                    adj_norm=adj_norm, device=device)
print(f'GNN predictions: {preds_gnn}')

print('\nSmoke test passed: forward pass, backward pass, and predict() all run without errors.')

## Full training (run on remote server)

```bash
cd /home/agung/Code/datathon/datathon-kaggle-task
uv run python3 -c "
import sys; sys.path.insert(0, 'task2')
import torch, numpy as np
from pathlib import Path
from src import load_articles, load_categories, load_states_train, load_states_test, load_or_build_adjacency
from src.dl_model import MLPScorer, GCNScorer, CandidateDataset, build_normalized_adj, predict
from torch.utils.data import DataLoader

BASE = Path('.'); CACHE = BASE / '.cache'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
articles = load_articles(); categories = load_categories()
train = load_states_train(); test = load_states_test(); adj = load_or_build_adjacency()
embeddings = np.load(CACHE / 'article_embeddings.npy')
all_cats = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(all_cats)}
cat_enc = np.zeros((len(articles), len(all_cats)), dtype=np.float32)
for _, r in categories.iterrows():
    cat_enc[r['article_id'], cat_to_idx[r['category']]] = 1.0

dataset = CandidateDataset(train, adj, embeddings, cat_enc)
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=4)
model = MLPScorer(emb_dim=384, n_cats=cat_enc.shape[1], hidden_dim=128).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = torch.nn.BCEWithLogitsLoss()

for epoch in range(5):
    model.train()
    total_loss = 0
    for batch in loader:
        curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats, out_deg, n_links, labels = [
            x.to(device) for x in batch
        ]
        scores = model(curr_emb, tgt_emb, cand_emb, curr_cats, tgt_cats, cand_cats, out_deg, n_links)
        loss = loss_fn(scores, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch}: loss = {total_loss/len(loader):.4f}')

preds = predict(model, test, adj, embeddings, cat_enc, device=device)
from src import make_submission
from datetime import datetime
sub_dir = BASE / 'submissions' / f'{datetime.now().strftime(\"%Y%m%d_%H%M%S\")}_dl_mlp'
sub_dir.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, preds, sub_dir / 'submission.csv')
torch.save(model.state_dict(), sub_dir / 'model.pt')
print(f'Saved: {sub_dir / "submission.csv"}')
" 2>&1
```

Same for GCNScorer (uses more memory due to dense adjacency).